## Import R libraries


In [ ]:
library(lme4)
library(mice)
library(arrow)
library(gt)
library(dplyr)
library(gtsummary)
library(arm)
library(ggplot2)
library(broom.mixed)
library(data.table)
library(lubridate)
library(future.apply)
library(patchwork)
library(rlang)
library(forcats)
library(performance)
library(patchwork)
library(mitml)

#Nick's packages
#packages <- c("tidyverse","ggthemes","styler","readxl","writexl","DBI","dbplyr","knitr","pandoc","janitor", "data.table", "duckdb","powerjoin","collapse","tidyfast","datapasta","fst","dtplyr","bit64","zoo","fuzzyjoin","arrow","hrbrthemes","here","table1", "rvest", "tidymodels", "pscl", "performance", "report")
packages <- c("future", "progressr", "ggthemes","merTools", "styler", "readxl", "writexl", "openxlsx", "labelled",  "webshot2", "flextable", "tableone", "readr", "DBI", "dbplyr", "pandoc", "janitor", "duckdb", "powerjoin", "collapse", "tidyfast", "datapasta", "fst", "dtplyr", "bit64", "zoo", "fuzzyjoin", "tigris", "hablar", "Hmisc", "table1", "arrow", "RColorBrewer", "viridis", "ggalluvial", "ggfittext", "tinytex", "ggrepel", "survival", "remotes", "ggforce", "tidyr", "tidytext", "Matrix", "lme4", "forestplot", "finalfit", "data.table", "broom.mixed", "car", "jtools", "DataExplorer", "psych", "sjPlot", "GGally", "mice", "haven", "dplyr", "gt", "tidyverse", "gtsummary", "performance", "report", "cluster" )

install_if_missing <- function(package) {
  if (!require(package, character.only = TRUE)) {
    install.packages(package, dependencies = TRUE)
    library(package, character.only = TRUE)
  }
}

sapply(packages, install_if_missing)
options(repr.matrix.max.cols = 1e3)    # Show more columns if needed


# Functions


In [ ]:
install_if_missing <- function(package) {
  if (!require(package, character.only = TRUE)) {
    install.packages(package, dependencies = TRUE)
    library(package, character.only = TRUE)
  }
}

prepare_data <- function(df, scale_vars, factor_vars, reference_levels = list(), means = original_means, sds = original_sds) {
  missing_scale  <- setdiff(scale_vars, names(df))
  missing_factor <- setdiff(factor_vars, names(df))
  if (length(missing_scale) > 0) {
    stop(sprintf('Missing scale vars in df: %s', paste(missing_scale, collapse = ', ')))
  }
  if (length(missing_factor) > 0) {
    stop(sprintf('Missing factor vars in df: %s', paste(missing_factor, collapse = ', ')))
  }
  for (var in scale_vars) {
    if (!is.null(means[[var]]) && !is.null(sds[[var]]) && var %in% names(df)) {
      df[[var]] <- scale(df[[var]], center = means[[var]], scale = sds[[var]])
    }
  }
  for (var in factor_vars) {
    df[[var]] <- factor(df[[var]])
    if (var %in% names(reference_levels)) {
      df[[var]] <- relevel(df[[var]], ref = reference_levels[[var]])
    }
  }
  return(df)
}

random_effect_aor <- function(
  data,
  re_group_name,
  aor_col_name,
  aor_ci_low_name,
  aor_ci_up_name,
  plot_title,
  x_axis_name,
  figure_name,
  y_min = NULL,
  y_max = NULL) {
  re_group <- rlang::sym(re_group_name)
  aor_col <- rlang::sym(aor_col_name)
  ci_lower_col <- rlang::sym(aor_ci_low_name)
  ci_upper_col <- rlang::sym(aor_ci_up_name)
  data <- data[order(data[[aor_col_name]]), ]
  data[[re_group_name]] <- factor(data[[re_group_name]], levels = data[[re_group_name]])
  data$color <- ifelse(data[[aor_col_name]] >= 1, '#2ca02c', '#dc3912')
  p <- ggplot2::ggplot(data, ggplot2::aes(x = !!re_group, y = !!aor_col)) +
    ggplot2::geom_errorbar(ggplot2::aes(ymin = !!ci_lower_col, ymax = !!ci_upper_col), width = 0.2, color = 'gray') +
    ggplot2::geom_point(shape = 21, fill = data$color, color = 'black') +
    ggplot2::geom_hline(yintercept = 1, linetype = "dashed", color = "blue") +
    ggplot2::labs(title = plot_title, x = x_axis_name, y = 'Adjusted Odds Ratio') +
    ggplot2::theme_minimal(base_size = 14) +
    ggplot2::theme(axis.text.x = ggplot2::element_blank(),
                   axis.ticks.x = ggplot2::element_blank(),
                   plot.title = ggplot2::element_text(hjust = 0.5)) +
    ggplot2::guides(color = "none")
  if (!is.null(y_min) || !is.null(y_max)) {
    p <- p + ggplot2::coord_cartesian(ylim = c(y_min, y_max))
  }
  return(p)
}

ltvv_obs_or_set_proportion_plot <- function(data) {
  df <- data
  bins <- c(-Inf, 8, 9, 10, 11, 12, Inf)
  labels <- c('≤8 cc/kg', '8-9 cc/kg', '9-10 cc/kg', '10-11 cc/kg', '11-12 cc/kg', '>12 cc/kg')
  df <- df %>% dplyr::mutate(`Percent of Tidal Volume / IBW (cc/kg)` = cut(`tidal_volume_set_or_obs_ibw`, breaks = bins, labels = labels, right = TRUE, include.lowest = TRUE))
  grouped <- df %>%
    dplyr::group_by(prov_npi_shifted, `Percent of Tidal Volume / IBW (cc/kg)`) %>%
    dplyr::summarise(count = dplyr::n(), .groups = 'drop') %>%
    dplyr::group_by(prov_npi_shifted) %>%
    dplyr::mutate(total = sum(count), proportion = count / total * 100) %>%
    dplyr::ungroup()
  plot_data <- grouped %>%
    dplyr::select(prov_npi_shifted, `Percent of Tidal Volume / IBW (cc/kg)`, proportion) %>%
    tidyr::pivot_wider(names_from = `Percent of Tidal Volume / IBW (cc/kg)`, values_from = proportion, values_fill = 0)
  sort_column <- labels[1]
  plot_data <- plot_data %>% dplyr::arrange(.data[[sort_column]])
  long_plot_data <- plot_data %>%
    tidyr::pivot_longer(-prov_npi_shifted, names_to = "TidalVolumeBin", values_to = "Proportion") %>%
    dplyr::mutate(prov_npi_shifted = factor(prov_npi_shifted, levels = unique(plot_data$prov_npi_shifted)),
                  TidalVolumeBin = factor(TidalVolumeBin, levels = rev(labels)))
  colors <- c("#dc3912", "#7b3294", "#377eb8", '#f39c12', "#f4d03f", "#2ca02c")
  p <- ggplot2::ggplot(long_plot_data, ggplot2::aes(y = prov_npi_shifted, x = Proportion, fill = TidalVolumeBin)) +
    ggplot2::geom_bar(stat = "identity", position = "fill", color = NA) +
    ggplot2::scale_fill_manual(values = colors, name = "Percent of Tidal Volume / IBW (cc/kg)") +
    ggplot2::guides(fill = ggplot2::guide_legend(reverse = TRUE)) +
    ggplot2::scale_x_continuous(labels = scales::percent_format(accuracy = 1)) +
    ggplot2::theme_minimal(base_size = 14) +
    ggplot2::theme(axis.text.y = ggplot2::element_blank(),
                   axis.ticks.y = ggplot2::element_blank(),
                   panel.grid.major.y = ggplot2::element_blank(),
                   panel.grid.minor.y = ggplot2::element_blank(),
                   legend.position = "top",
                   legend.title = ggplot2::element_text(size = 12),
                   legend.text = ggplot2::element_text(size = 10, hjust = 0),
                   legend.key.size = grid::unit(0.8, "lines"),
                   legend.spacing.y = grid::unit(0.5, "lines"),
                   legend.box.spacing = grid::unit(0.5, "lines"),
                   plot.title = ggplot2::element_text(hjust = 0.5)) +
    ggplot2::labs(y = 'Provider', x = 'Proportion of Set or Obs Tidal Volume (%)', title = 'Proportion of Set or Obs Tidal Volume by Provider')
  return(p)
}

ltvv8_proportion_plot <- function(data, plot_title = 'Proportion of Set Tidal Volume by Provider') {
  df <- data
  bins <- c(-Inf, 8, 9, 10, 11, 12, Inf)
  labels <- c('\u22648 cc/kg', '8-9 cc/kg', '9-10 cc/kg', '10-11 cc/kg', '11-12 cc/kg', '>12 cc/kg')
  df <- df %>% dplyr::mutate(`Percent of Tidal Volume / IBW (cc/kg)` = cut(`tidal_volume_set_ibw`, breaks = bins, labels = labels, right = TRUE, include.lowest = TRUE))
  grouped <- df %>%
    dplyr::group_by(prov_npi_shifted, `Percent of Tidal Volume / IBW (cc/kg)`) %>%
    dplyr::summarise(count = dplyr::n(), .groups = 'drop') %>%
    dplyr::group_by(prov_npi_shifted) %>%
    dplyr::mutate(total = sum(count), proportion = count / total * 100) %>%
    dplyr::ungroup()
  plot_data <- grouped %>%
    dplyr::select(prov_npi_shifted, `Percent of Tidal Volume / IBW (cc/kg)`, proportion) %>%
    tidyr::pivot_wider(names_from = `Percent of Tidal Volume / IBW (cc/kg)`, values_from = proportion, values_fill = 0)
  sort_column <- labels[1]
  plot_data <- plot_data %>% dplyr::arrange(.data[[sort_column]])
  long_plot_data <- plot_data %>%
    tidyr::pivot_longer(-prov_npi_shifted, names_to = "TidalVolumeBin", values_to = "Proportion") %>%
    dplyr::mutate(prov_npi_shifted = factor(prov_npi_shifted, levels = unique(plot_data$prov_npi_shifted)),
                  TidalVolumeBin = factor(TidalVolumeBin, levels = rev(labels)))
  colors <- c("#dc3912", "#7b3294", "#377eb8", '#f39c12', "#f4d03f", "#2ca02c")
  p <- ggplot2::ggplot(long_plot_data, ggplot2::aes(y = prov_npi_shifted, x = Proportion, fill = TidalVolumeBin)) +
    ggplot2::geom_bar(stat = "identity", position = "fill", color = NA) +
    ggplot2::scale_fill_manual(values = colors, name = "Percent of Tidal Volume / IBW (cc/kg)") +
    ggplot2::guides(fill = ggplot2::guide_legend(reverse = TRUE)) +
    ggplot2::scale_x_continuous(labels = scales::percent_format(accuracy = 1)) +
    ggplot2::theme_minimal(base_size = 14) +
    ggplot2::theme(axis.text.y = ggplot2::element_blank(),
                   axis.ticks.y = ggplot2::element_blank(),
                   panel.grid.major.y = ggplot2::element_blank(),
                   panel.grid.minor.y = ggplot2::element_blank(),
                   legend.position = "top",
                   legend.title = ggplot2::element_text(size = 12),
                   legend.text = ggplot2::element_text(size = 10, hjust = 0),
                   legend.key.size = grid::unit(0.8, "lines"),
                   legend.spacing.y = grid::unit(0.5, "lines"),
                   legend.box.spacing = grid::unit(0.5, "lines"),
                   plot.title = ggplot2::element_text(hjust = 0.5)) +
    ggplot2::labs(y = 'Provider', x = 'Proportion of Set Tidal Volume (%)', title = plot_title)
  return(p)
}

# Variant: bins focused on ≤6 cc/kg
ltvv6_proportion_plot <- function(data, plot_title = 'Proportion of Set Tidal Volume by Provider') {
  df <- data
  bins <- c(-Inf, 6, 7, 8, 10, 12, Inf)
  labels <- c('\u22646 cc/kg', '6-7 cc/kg', '7-8 cc/kg', '8-10 cc/kg', '10-12 cc/kg', '>12 cc/kg')
  df <- df %>% dplyr::mutate(`Percent of Tidal Volume / IBW (cc/kg)` = cut(`tidal_volume_set_ibw`, breaks = bins, labels = labels, right = TRUE, include.lowest = TRUE))
  grouped <- df %>%
    dplyr::group_by(prov_npi_shifted, `Percent of Tidal Volume / IBW (cc/kg)`) %>%
    dplyr::summarise(count = dplyr::n(), .groups = 'drop') %>%
    dplyr::group_by(prov_npi_shifted) %>%
    dplyr::mutate(total = sum(count), proportion = count / total * 100) %>%
    dplyr::ungroup()
  plot_data <- grouped %>%
    dplyr::select(prov_npi_shifted, `Percent of Tidal Volume / IBW (cc/kg)`, proportion) %>%
    tidyr::pivot_wider(names_from = `Percent of Tidal Volume / IBW (cc/kg)`, values_from = proportion, values_fill = 0)
  sort_column <- labels[1]
  plot_data <- plot_data %>% dplyr::arrange(.data[[sort_column]])
  long_plot_data <- plot_data %>%
    tidyr::pivot_longer(-prov_npi_shifted, names_to = "TidalVolumeBin", values_to = "Proportion") %>%
    dplyr::mutate(prov_npi_shifted = factor(prov_npi_shifted, levels = unique(plot_data$prov_npi_shifted)),
                  TidalVolumeBin = factor(TidalVolumeBin, levels = rev(labels)))
  colors <- c("#dc3912", "#7b3294", "#377eb8", '#f39c12', "#f4d03f", "#2ca02c")
  p <- ggplot2::ggplot(long_plot_data, ggplot2::aes(y = prov_npi_shifted, x = Proportion, fill = TidalVolumeBin)) +
    ggplot2::geom_bar(stat = "identity", position = "fill", color = NA) +
    ggplot2::scale_fill_manual(values = colors, name = "Percent of Tidal Volume / IBW (cc/kg)") +
    ggplot2::guides(fill = ggplot2::guide_legend(reverse = TRUE)) +
    ggplot2::scale_x_continuous(labels = scales::percent_format(accuracy = 1)) +
    ggplot2::theme_minimal(base_size = 14) +
    ggplot2::theme(axis.text.y = ggplot2::element_blank(),
                   axis.ticks.y = ggplot2::element_blank(),
                   panel.grid.major.y = ggplot2::element_blank(),
                   panel.grid.minor.y = ggplot2::element_blank(),
                   legend.position = "top",
                   legend.title = ggplot2::element_text(size = 12),
                   legend.text = ggplot2::element_text(size = 10, hjust = 0),
                   legend.key.size = grid::unit(0.8, "lines"),
                   legend.spacing.y = grid::unit(0.5, "lines"),
                   legend.box.spacing = grid::unit(0.5, "lines"),
                   plot.title = ggplot2::element_text(hjust = 0.5)) +
    ggplot2::labs(y = 'Provider', x = 'Proportion of Set Tidal Volume (%)', title = plot_title)
  return(p)
}

pool_fixed_effects <- function(model_obj) {
  mitml_res <- mitml::testEstimates(model = model_obj, extra.pars = TRUE)
  fe <- mitml_res$estimates
  data.frame(
    Term     = rownames(fe),
    OR       = exp(fe[, "Estimate"]),
    Lower_CI = exp(fe[, "Estimate"] - 1.96 * fe[, "Std.Error"]),
    Upper_CI = exp(fe[, "Estimate"] + 1.96 * fe[, "Std.Error"]),
    P_value  = fe[, "P(>|t|)"],
    row.names = NULL
  )
}

create_fe_table <- function(fe_df,
                            title,
                            output_file = NULL,
                            drop_intercept = TRUE,
                            term_map = NULL,
                            digits_or = 2,
                            digits_ci = 2) {
  stopifnot(is.data.frame(fe_df),
            all(c("Term","OR","Lower_CI","Upper_CI","P_value") %in% names(fe_df)))
  df <- fe_df
  if (drop_intercept) {
    df <- df %>% dplyr::filter(!grepl("^\\(Intercept\\)$", Term))
  }
  if (is.null(term_map) && exists('rename_terms')) {
    term_map <- rename_terms
  }
  if (!is.null(term_map)) {
    df <- df %>% dplyr::mutate(Term = ifelse(Term %in% names(term_map), term_map[Term], Term))
  }
  if (exists('custom_order')) {
    df <- df %>% dplyr::mutate(Term = factor(Term, levels = custom_order)) %>% dplyr::arrange(Term) %>% dplyr::mutate(Term = as.character(Term))
  }
  df_fmt <- df %>%
    dplyr::mutate(
      `OR (95% CI)` = sprintf(paste0("%.", digits_or, "f (%.", digits_ci, "f, %.", digits_ci, "f)"), OR, Lower_CI, Upper_CI),
      `p-value` = gtsummary::style_pvalue(P_value)
    ) %>%
    dplyr::select(Term, `OR (95% CI)`, `p-value`)
  tab <- df_fmt %>%
    gt::gt(rowname_col = NULL) %>%
    gt::tab_header(title = gt::md(title)) %>%
    gt::fmt_markdown(columns = dplyr::everything()) %>%
    gt::tab_options(table.font.size = gt::px(14)) %>%
    gt::opt_row_striping() %>%
    gt::opt_vertical_padding(scale = 0.5)
  if (!is.null(output_file)) {
    html_code <- gt::as_raw_html(tab)
    writeLines(html_code, output_file)
    message("Table saved to: ", output_file)
  }
  tab
}

create_forest_plot <- function(data,
                               main_breaks = c(0, 2, 4, 6),
                               main_limits = c(0, 6),
                               year_breaks = c(0, 5, 10, 15, 20),
                               year_limits = c(0, 20),
                               filename = "fixed_effects.tiff",
                               width = 8,
                               height = 4,
                               dpi = 600) {
  plot_data <- data |>
    dplyr::filter(Term != "(Intercept)") |>
    dplyr::rename(term = Term, OR = OR, CI_lower = `Lower_CI`, CI_upper = `Upper_CI`) |>
    dplyr::mutate(term_chr = as.character(term),
                  mapped = unname(rename_terms[term_chr]),
                  term_pretty = dplyr::coalesce(mapped, term_chr),
                  plot_group = ifelse(grepl("^Year:", term_pretty), "Year Effects", "Main Predictors")) |>
    dplyr::select(-mapped) |>
    dplyr::mutate(term_pretty = factor(term_pretty, levels = rev(intersect(custom_order, unique(term_pretty)))))
  main_effects <- plot_data %>% dplyr::filter(plot_group == "Main Predictors")
  year_effects <- plot_data %>% dplyr::filter(plot_group == "Year Effects")
  plot_main <- ggplot2::ggplot(main_effects, ggplot2::aes(x = term_pretty, y = OR)) +
    ggplot2::geom_point(size = 2) +
    ggplot2::geom_errorbar(ggplot2::aes(ymin = CI_lower, ymax = CI_upper), width = 0.2) +
    ggplot2::geom_hline(yintercept = 1, linetype = "dashed", color = "black") +
    ggplot2::coord_flip(clip = "off") +
    ggplot2::scale_y_continuous(breaks = main_breaks, limits = main_limits, expand = ggplot2::expansion(mult = c(0, 0))) +
    ggplot2::labs(title = "Patient and Hospital", x = "Fixed Effect", y = "Odds Ratio") +
    ggplot2::theme_minimal()
  plot_years <- ggplot2::ggplot(year_effects, ggplot2::aes(x = term_pretty, y = OR)) +
    ggplot2::geom_point(size = 2) +
    ggplot2::geom_errorbar(ggplot2::aes(ymin = CI_lower, ymax = CI_upper), width = 0.2) +
    ggplot2::geom_hline(yintercept = 1, linetype = "dashed", color = "black") +
    ggplot2::coord_flip(clip = "off") +
    ggplot2::scale_y_continuous(breaks = year_breaks, limits = year_limits, expand = ggplot2::expansion(mult = c(0, 0))) +
    ggplot2::labs(title = "Year", x = "Fixed Effect", y = "Odds Ratio") +
    ggplot2::theme_minimal()
  combined_plot <- plot_main + plot_years + patchwork::plot_layout(nrow = 1) + patchwork::plot_annotation(tag_levels = 'A')
  ggplot2::ggsave(filename, plot = combined_plot, dpi = dpi, width = width, height = height)
  message("Plot saved to: ", filename)
  return(combined_plot)
}

extract_random_effects <- function(model) {
  ranefs <- lme4::ranef(model)$prov_npi_shifted
  se_physician <- attr(lme4::ranef(model, condVar = TRUE)$prov_npi_shifted, "postVar")[1, 1, ]
  aor <- exp(ranefs[, "(Intercept)"])
  ci_lower <- exp(ranefs[, "(Intercept)"] - 1.96 * sqrt(se_physician))
  ci_upper <- exp(ranefs[, "(Intercept)"] + 1.96 * sqrt(se_physician))
  data.frame(prov_npi_shifted = rownames(ranefs), AOR = aor, CI_lower = ci_lower, CI_upper = ci_upper)
}

ltvv_median_iqr <- function(data, adherence_bins = c("<8 cc/kg")) {
  df <- data
  bins <- c(-Inf, 8, 9, 10, 11, 12, Inf)
  labels <- c('≤8 cc/kg', '8-9 cc/kg', '9-10 cc/kg', '10-11 cc/kg', '11-12 cc/kg', '>12 cc/kg')
  df <- df %>% dplyr::mutate(`Percent of Tidal Volume / IBW (cc/kg)` = cut(`tidal_volume_set_ibw`, breaks = bins, labels = labels, right = TRUE, include.lowest = TRUE))
  grouped <- df %>%
    dplyr::group_by(.data$prov_npi_shifted, .data$`Percent of Tidal Volume / IBW (cc/kg)`) %>%
    dplyr::summarise(count = dplyr::n(), .groups = "drop") %>%
    dplyr::group_by(.data$prov_npi_shifted) %>%
    dplyr::mutate(total = sum(.data$count), proportion = count / total * 100) %>%
    dplyr::ungroup()
  plot_data <- grouped %>%
    dplyr::select(.data$prov_npi_shifted, .data$`Percent of Tidal Volume / IBW (cc/kg)`, .data$proportion) %>%
    tidyr::pivot_wider(names_from = `Percent of Tidal Volume / IBW (cc/kg)`, values_from = proportion, values_fill = 0)
  adherence_bins <- intersect(adherence_bins, colnames(plot_data))
  adherence <- rowSums(dplyr::select(plot_data, dplyr::all_of(adherence_bins)), na.rm = TRUE)
  med <- stats::median(adherence, na.rm = TRUE)
  q <- stats::quantile(adherence, probs = c(0.25, 0.75), na.rm = TRUE, type = 7)
  message(sprintf("Provider LTVV adherence (%% in %s): median %.1f%% [IQR %.1f?%.1f%%] (n=%d providers)", paste(adherence_bins, collapse = " + "), med, q[[1]], q[[2]], nrow(plot_data)))
  return(tibble::tibble(n_providers = nrow(plot_data), adherence_bins = paste(adherence_bins, collapse = " + "), median = med, q1 = q[[1]], q3 = q[[2]]))
}

summarize_model <- function(model_list, data, outcome, model_name, random_effect_var = "prov_npi_shifted") {
  # --- pull model spec from first fitted model ---
  get_model_spec <- function(model) {
    f <- formula(model)
    # outcome (LHS)
    outcome_name <- all.vars(lme4::nobars(f)[[2]])
    # fixed-effect RHS (covariates)
    rhs_fixed <- lme4::nobars(f)[[3]]
    covariates <- setdiff(all.vars(rhs_fixed), outcome_name)
    # random effects terms
    random_effects <- vapply(lme4::findbars(f), function(b) deparse(b[[3]]), character(1))

    list(
      outcome = outcome_name,
      random_effect = random_effects,
      covariates = covariates
    )
  }

  spec <- get_model_spec(model_list[[1]])

  # --- existing summary calculations ---
  log_mor_vec <- c()
  var_log_mor_vec <- c()
  for (model in model_list) {
    vc_df <- as.data.frame(VarCorr(model))
    variance_random_intercepts <- vc_df$vcov[1]
    sd_random_intercepts <- vc_df$sdcor[1]
    n_clusters <- nrow(lme4::ranef(model)[[random_effect_var]])
    sem_sd <- sd_random_intercepts / sqrt(n_clusters)
    var_sem <- 2 * variance_random_intercepts * (sem_sd^2)
    mor <- exp(sqrt(2 * variance_random_intercepts) * 0.6745)
    log_mor <- log(mor)
    deriv_log_mor <- 0.6745 / sqrt(2 * variance_random_intercepts)
    var_log_mor <- (deriv_log_mor^2) * var_sem
    log_mor_vec <- c(log_mor_vec, log_mor)
    var_log_mor_vec <- c(var_log_mor_vec, var_log_mor)
  }
  m <- length(log_mor_vec)
  Q_bar <- mean(log_mor_vec)
  W <- mean(var_log_mor_vec)
  B <- var(log_mor_vec)
  Tval <- W + (1 + 1/m) * B
  df <- (m - 1) * (1 + (W / ((1 + 1/m) * B)))^2
  crit <- stats::qt(0.975, df)
  log_mor_lower <- Q_bar - crit * sqrt(Tval)
  log_mor_upper <- Q_bar + crit * sqrt(Tval)
  mor_pooled <- exp(Q_bar)
  mor_lower <- exp(log_mor_lower)
  mor_upper <- exp(log_mor_upper)

  icc_vec <- c()
  var_icc_vec <- c()
  for (model in model_list) {
    var_re <- as.numeric(as.data.frame(VarCorr(model))$vcov[1])
    var_res <- pi^2 / 3
    icc <- var_re / (var_re + var_res)
    n_clusters <- nrow(lme4::ranef(model)[[random_effect_var]])
    se_icc <- sqrt((2 * icc^2 * (1 - icc)^2) / (n_clusters - 1))
    icc_vec <- c(icc_vec, icc)
    var_icc_vec <- c(var_icc_vec, se_icc^2)
  }
  m_i <- length(icc_vec)
  Qb_i <- mean(icc_vec)
  W_i <- mean(var_icc_vec)
  B_i <- var(icc_vec)
  T_i <- W_i + (1 + 1/m_i) * B_i
  df_i <- (m_i - 1) * (1 + (W_i / ((1 + 1/m_i) * B_i)))^2
  crit_i <- stats::qt(0.975, df_i)
  icc_lower <- Qb_i - crit_i * sqrt(T_i)
  icc_upper <- Qb_i + crit_i * sqrt(T_i)
  icc_pooled <- Qb_i

  baseline_prob <- mean(as.numeric(as.character(data[[outcome]])))
  baseline_odds <- baseline_prob / (1 - baseline_prob)
  new_odds <- baseline_odds * mor_pooled
  new_prob <- new_odds / (1 + new_odds)
  absolute_risk_diff <- new_prob - baseline_prob
  NNT <- ifelse(absolute_risk_diff == 0, Inf, abs(1 / absolute_risk_diff))

  n_hospitalizations <- length(unique(data$hospitalizations_joined_id))
  n_patients <- length(unique(data$mdm_link_id))
  n_providers <- length(unique(data[[random_effect_var]]))
  n_observations <- nrow(data)

  summary <- list(
    model_name = model_name,
    # model spec (from formula)
    outcome = spec$outcome,
    random_effect = spec$random_effect,
    covariates = spec$covariates,
    # existing metrics
    icc = icc_pooled,
    icc_ci_lower = icc_lower,
    icc_ci_upper = icc_upper,
    mor = mor_pooled,
    mor_ci_lower = mor_lower,
    mor_ci_upper = mor_upper,
    baseline_prob = baseline_prob,
    new_prob = new_prob,
    absolute_risk_diff = absolute_risk_diff,
    nnt = NNT,
    n_hospitalizations = n_hospitalizations,
    n_patients = n_patients,
    n_providers = n_providers,
    n_observations = n_observations
  )
  return(summary)
}

create_model_summary_html <- function(model_summaries, output_file = "model_summary.html") {
  if (inherits(model_summaries, "model_summary")) {
    model_summaries <- list(model_summaries)
  }
  stopifnot(is.list(model_summaries), length(model_summaries) >= 1)

  fmt_pct <- function(x, digits = 1) sprintf(paste0("%.", digits, "f%%"), 100 * x)
  fmt_num <- function(x, digits = 3) sprintf(paste0("%.", digits, "f"), x)
  fmt_ci  <- function(est, lo, hi, digits = 3) sprintf(paste0("%.", digits, "f (%.", digits, "f - %.", digits, "f)"), est, lo, hi)
  fmt_int <- function(x) format(x, big.mark = ",", scientific = FALSE)
  fmt_nnt <- function(x) ifelse(is.finite(x), sprintf("%.1f", x), "&infin;")

  html_content <- paste0(
'<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8">
  <title>Model Summary Report</title>
  <style>
    body { font-family: Arial, sans-serif; margin: 40px; color: #2c3e50; }
    h1 { color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 8px; }
    h2 { color: #34495e; margin-top: 28px; }
    h3 { color: #34495e; margin-top: 18px; }
    table { border-collapse: collapse; width: 100%; margin: 16px 0; }
    th, td { border: 1px solid #ddd; padding: 10px; text-align: left; }
    th { background-color: #f4f6f8; font-weight: bold; }
    tr:nth-child(even) { background-color: #fafbfc; }
    .metric { font-weight: bold; color: #2980b9; }
    .summary-box { background-color: #ecf0f1; padding: 14px; margin: 12px 0; border-radius: 6px; }
    .small { color: #6b7280; font-size: 0.9em; }
  </style>
</head>
<body>
  <h1>Model Summary Report</h1>
  <p class="small"><strong>Generated on:</strong> ', format(Sys.time(), "%B %d, %Y %H:%M %Z"), '</p>
  <h2>Model Comparison Summary</h2>
  <table>
    <tr>
      <th>Model</th>
      <th>ICC (95% CI)</th>
      <th>MOR (95% CI)</th>
      <th>Baseline Risk</th>
      <th>Risk Difference</th>
      <th>NNT</th>
      <th>Observations</th>
      <th>Providers</th>
    </tr>'
  )

  for (summary in model_summaries) {
    req <- c("model_name","icc","icc_ci_lower","icc_ci_upper",
             "mor","mor_ci_lower","mor_ci_upper",
             "baseline_prob","absolute_risk_diff","nnt",
             "n_observations","n_providers")
    missing_fields <- setdiff(req, names(summary))
    if (length(missing_fields) > 0) {
      stop(sprintf("Missing fields in a model summary: %s", paste(missing_fields, collapse = ", ")))
    }

    html_content <- paste0(html_content, sprintf('
    <tr>
      <td><strong>%s</strong></td>
      <td>%s</td>
      <td>%s</td>
      <td>%s</td>
      <td>%s</td>
      <td>%s</td>
      <td>%s</td>
      <td>%s</td>
    </tr>',
      summary$model_name,
      fmt_ci(summary$icc, summary$icc_ci_lower, summary$icc_ci_upper, digits = 3),
      fmt_ci(summary$mor, summary$mor_ci_lower, summary$mor_ci_upper, digits = 3),
      fmt_pct(summary$baseline_prob, 1),
      fmt_pct(summary$absolute_risk_diff, 2),
      fmt_nnt(summary$nnt),
      fmt_int(summary$n_observations),
      fmt_int(summary$n_providers)
    ))
  }

  html_content <- paste0(html_content, '
  </table>
')

  for (summary in model_summaries) {
    req2 <- c("model_name","icc","icc_ci_lower","icc_ci_upper",
              "mor","mor_ci_lower","mor_ci_upper",
              "baseline_prob","new_prob","absolute_risk_diff","nnt",
              "n_hospitalizations","n_patients","n_providers","n_observations")
    missing_fields <- setdiff(req2, names(summary))
    if (length(missing_fields) > 0) {
      stop(sprintf("Missing fields in a model summary (detailed section): %s", paste(missing_fields, collapse = ", ")))
    }

    req3 <- c("outcome","random_effect","covariates")
    missing_fields <- setdiff(req3, names(summary))
    if (length(missing_fields) > 0) {
      stop(sprintf("Missing model spec fields: %s", paste(missing_fields, collapse = ", ")))
    }

    html_content <- paste0(html_content, sprintf('
  <h2>%s - Detailed Results</h2>
  <div class="summary-box">
    <h3>Model Specification</h3>
    <p><span class="metric">Outcome:</span> %s</p>
    <p><span class="metric">Random Effect:</span> %s</p>
    <p><span class="metric">Covariates:</span> %s</p>
  </div>
  <div class="summary-box">
    <h3>Intraclass Correlation Coefficient (ICC)</h3>
    <p><span class="metric">Pooled ICC:</span> %s</p>

    <h3>Median Odds Ratio (MOR)</h3>
    <p><span class="metric">Pooled MOR:</span> %s</p>

    <h3>Risk Assessment</h3>
    <p><span class="metric">Baseline Probability:</span> %s (%s)</p>
    <p><span class="metric">New Probability with MOR:</span> %s (%s)</p>
    <p><span class="metric">Absolute Risk Difference:</span> %s</p>
    <p><span class="metric">Number Needed to Treat (NNT):</span> %s</p>

    <h3>Dataset Summary</h3>
    <p><span class="metric">Hospitalizations:</span> %s</p>
    <p><span class="metric">Patients:</span> %s</p>
    <p><span class="metric">Providers:</span> %s</p>
    <p><span class="metric">Total Observations:</span> %s</p>
  </div>',
      summary$model_name,
      summary$outcome,
      paste(summary$random_effect, collapse = ", "),
      paste(summary$covariates, collapse = ", "),
      fmt_ci(summary$icc, summary$icc_ci_lower, summary$icc_ci_upper, digits = 3),
      fmt_ci(summary$mor, summary$mor_ci_lower, summary$mor_ci_upper, digits = 3),
      fmt_num(summary$baseline_prob, 3), fmt_pct(summary$baseline_prob, 1),
      fmt_num(summary$new_prob, 3), fmt_pct(summary$new_prob, 1),
      fmt_pct(summary$absolute_risk_diff, 2),
      fmt_nnt(summary$nnt),
      fmt_int(summary$n_hospitalizations),
      fmt_int(summary$n_patients),
      fmt_int(summary$n_providers),
      fmt_int(summary$n_observations)
    ))
  }

  html_content <- paste0(html_content, '
</body>
</html>')

  writeLines(html_content, output_file)
  message("HTML summary saved to: ", output_file)
  invisible(html_content)
}

<h2> Rename and reorder terms </h2>

In [ ]:
# Create a mapping from current term names to display names
rename_terms <- c(
  "(Intercept)" = "(Intercept)",
  "age" = "Age",
  
  "race_categoryAmerican Indian or Alaska Native" = "Race: American Indian or Alaska Native",
  "race_categoryAsian" = "Race: Asian",
  "race_categoryBlack or African-American" = "Race: Black or African-American",
  "race_categoryBlack or African American" = "Race: Black or African-American",
  "race_categoryNative Hawaiian or Other Pacific Islander" = "Race: Native Hawaiian or Other Pacific Islander",
  "race_categoryOther" = "Race: Other",
  "race_categoryUnknown" = "Race: Unknown",
  
  "sex_categoryMale" = "Sex: Male",
  "height_cm" = "Height (cm)",
  
  "recorded_year2012" = "Year: 2012",
  "recorded_year2013" = "Year: 2013",
  "recorded_year2014" = "Year: 2014",
  "recorded_year2015" = "Year: 2015",
  "recorded_year2016" = "Year: 2016",
  "recorded_year2017" = "Year: 2017",
  "recorded_year2018" = "Year: 2018",
  "recorded_year2019" = "Year: 2019",
  "recorded_year2020" = "Year: 2020",
  "recorded_year2021" = "Year: 2021",
  "recorded_year2022" = "Year: 2022",
  "recorded_year2023" = "Year: 2023",
  "recorded_year2024" = "Year: 2024",
  "recorded_year2025" = "Year: 2025",
  
  "elix_vw" = "Elixhauser Score",
  
  "hospital_idHospital 1" = "Hospital 1",
  "hospital_idHospital 2" = "Hospital 2",
  "hospital_idHospital 3" = "Hospital 3",
  "hospital_idHospital 4" = "Hospital 4",
  "hospital_idHospital 5" = "Hospital 5",
  "hospital_idHospital 6" = "Hospital 6",
  "hospital_idHospital 7" = "Hospital 7",
  "hospital_idHospital 8" = "Hospital 8",
  "hospital_idHospital 9" = "Hospital 9",
  
  "bmi_calc" = "BMI",
  "sf_ratio" = "SF Ratio",
  "pco2" = "PCO2",
  "ph" = "pH",
  "laps2" = "LAPS2 Score",
  "covidTRUE" = 'SARS-CoV-2'
)

custom_order <- c(
  "(Intercept)",
  "Age",
  "BMI",
  
  "Race: American Indian or Alaska Native",
  "Race: Asian",
  "Race: Black or African-American",
  "Race: Native Hawaiian or Other Pacific Islander",
  "Race: Other",
  "Race: Unknown",
  
  "Sex: Male",
  "Height (cm)",
  
  "Elixhauser Score",
  "LAPS2 Score",

  "SF Ratio",
  "PCO2",
  "pH",

  "SARS-CoV-2",
    
  "Hospital 1",
  "Hospital 2",
  "Hospital 3",
  "Hospital 4",
  "Hospital 5",
  "Hospital 6",
  "Hospital 7",
  "Hospital 8",
  "Hospital 9",

  "Year: 2012",
  "Year: 2013",
  "Year: 2014",
  "Year: 2015",
  "Year: 2016",
  "Year: 2017",
  "Year: 2018",
  "Year: 2019",
  "Year: 2020",
  "Year: 2021",
  "Year: 2022",
  "Year: 2023",
  "Year: 2024",
  "Year: 2025"
)

<h2> Read in LPV full dataset </h2>


In [ ]:
data <- read_parquet('intermediate_outputs/LPV_final_data.parquet')

# Print counts
print(length(unique(data$hospitalizations_joined_id)))
print(length(unique(data$prov_npi_shifted)))
print(nrow(data))
# head(data,10)

<h3> MICE imputation with Rubin's Rules </h3>

In [ ]:
# Variable definitions for scaling and factor levels
scale_vars <- c('age', 'bmi_calc', 'height_cm', 'elix_vw', 'laps2', 'ph', 'pco2', 'sf_ratio')

# Note: dataset has ltvv_6/ltvv_8 and ltvv_tidal_volume_set_or_obs_6/_8 (not the generic name)
factor_vars <- c('ltvv_6', 'ltvv_8', 'ltvv_tidal_volume_set_or_obs_6', 'ltvv_tidal_volume_set_or_obs_8',
  'race_category', 'sex_category', 'recorded_year', 'hospital_id', 'icu_type',
  'prov_npi_shifted', 'mdm_link_id', 'covid')

reference_levels <- list(
  race_category = 'White',
  sex_category  = 'Female',
  hospital_id   = 'Hospital Ref',
  icu_type      = 'Mixed'
)

In [ ]:
# 1. Define your imputation variables
vars_for_impute <- c("age", "sex_category", "height_cm", "bmi_calc", "elix_vw", "laps2", "ph", "pco2", "sf_ratio", "hospital_id")
# Split into imputed and non-imputed data
data_impute <- data[, vars_for_impute]
data_nonimpute <- data[, !(names(data) %in% vars_for_impute)]

# Get original means and SDs for imputation variables - need them for scaling after imputation
original_means <- sapply(data[scale_vars], function(x) mean(x, na.rm = TRUE))
original_sds   <- sapply(data[scale_vars], function(x) sd(x, na.rm = TRUE))

print(length(rownames(data_impute)))
print(summary(data_impute))
print(sapply(data_impute, function(x) mean(is.na(x)) * 100))

In [ ]:
# If variable needs to be factorized, do so prior to imputation
for (var in factor_vars) {
  if (var %in% names(data_impute)) {
    data_impute[[var]] <- factor(data_impute[[var]])
  }
}

# 3. Run MICE only on imputation variables
imp <- mice(data_impute, m = 5, method = "pmm", seed = 123)

# 4. Combine imputed datasets with non-imputed columns
completed_data_list <- lapply(1:5, function(i) {
  data_combined <- cbind(complete(imp, i), data_nonimpute)
  prepare_data(
    data_combined, 
    scale_vars, 
    factor_vars, 
    reference_levels, 
    means = original_means, 
    sds = original_sds
  )  # Apply scaling + factor transformations
})

head(completed_data_list[[1]], 5)

<h1> Create datasets for each model </h1>

In [ ]:
# Define cohorts from imputed datasets (list of 5)
mv_data_list <- completed_data_list

ards_data_list <- lapply(completed_data_list, function(df) {
  df %>% dplyr::filter(cohort_eligible == 1L)
})

# Wrap in mitml.list for downstream pooled modeling
mv_data  <- as.mitml.list(mv_data_list)
ards_data <- as.mitml.list(ards_data_list)

cat(sprintf('Total ards datasets: %d (each aligned to completed_data_list)\n', length(ards_data)))

In [ ]:
# Derive initial and subsequent datasets for MV and ards (lists of 5)
initial_mv_data_list <- lapply(mv_data, function(df) {
  df %>% dplyr::filter(initial_vent_day == TRUE)
})

subsequent_mv_data_list <- lapply(mv_data, function(df) {
  df %>% dplyr::filter(initial_vent_day == FALSE)
})

initial_ards_data_list <- lapply(ards_data, function(df) {
  df %>% dplyr::filter(initial_vent_day == TRUE)
})

subsequent_ards_data_list <- lapply(ards_data, function(df) {
  df %>% dplyr::filter(initial_vent_day == FALSE)
})

initial_mv_data       <- as.mitml.list(initial_mv_data_list)
subsequent_mv_data    <- as.mitml.list(subsequent_mv_data_list)
initial_ards_data      <- as.mitml.list(initial_ards_data_list)
subsequent_ards_data   <- as.mitml.list(subsequent_ards_data_list)

cat(sprintf('Initial MV datasets: %d; Subsequent MV datasets: %d\n', length(initial_mv_data), length(subsequent_mv_data)))
cat(sprintf('Initial ards datasets: %d; Subsequent ards datasets: %d\n', length(initial_ards_data), length(subsequent_ards_data)))
cat(sprintf('Rows in initial MV dataset 1: %d\n', nrow(initial_mv_data[[1]])))
cat(sprintf('Rows in subsequent MV dataset 1: %d\n', nrow(subsequent_mv_data[[1]])))
cat(sprintf('Rows in initial ards dataset 1: %d\n', nrow(initial_ards_data[[1]])))
cat(sprintf('Rows in subsequent ards dataset 1: %d\n', nrow(subsequent_ards_data[[1]])))


In [ ]:
ards6_data <- ards_data
ards8_data <- ards_data

<h1> Figures 2 and 3 </h1>

-  ards6 - Tidal Volume Set by Provider - two panel
-  ards6 - Fixed effects plot

<h2> Fit ards6 model </h2>

In [ ]:
# Cap/raise threads per worker; try 2 to lift CPU a bit
Sys.setenv(
  OMP_NUM_THREADS = "2",
  MKL_NUM_THREADS = "2",
  OPENBLAS_NUM_THREADS = "2",
  VECLIB_MAXIMUM_THREADS = "2",
  NUMEXPR_NUM_THREADS = "2"
)

# Use as many workers as helpful, capped by cores and list length
max_workers <- min(length(ards_data), parallelly::availableCores())  # if 5 items, this gives up to 5
future::plan(multisession, workers = max_workers)

In [ ]:
# Check all downstream variables for missingness
vars_all <- unique(c(
  "age","race_category","sex_category","height_cm","recorded_year","elix_vw",
  "hospital_id","icu_type","bmi_calc","sf_ratio","pco2","ph","laps2","covid",
  "ltvv_6","ltvv_8","ltvv_tidal_volume_set_or_obs_6","ltvv_tidal_volume_set_or_obs_8",
  "prov_npi_shifted"
))

df <- mv_data[[1]]  # or ards_data[[1]]

na_counts <- sapply(df[vars_all], function(x) sum(is.na(x)))
na_pct    <- sapply(df[vars_all], function(x) mean(is.na(x)) * 100)

data.frame(var = vars_all, na = na_counts, na_pct = round(na_pct, 2))


In [ ]:
explanatory_vars <- c("age", "race_category", "sex_category", "height_cm", "recorded_year", "elix_vw", "hospital_id", "icu_type", 'bmi_calc',
                      'sf_ratio', 'pco2', 'ph', 'laps2')

random_effect <- "prov_npi_shifted"
outcome <- 'ltvv_6'


# Set model seed
set.seed(42) 
formula_text <- paste(outcome,"~", paste(explanatory_vars, collapse = " + "), " + (1 | ", random_effect, ")")
message(paste('Formula:',formula_text))

message('Fitting model...')
ards6_model <- future_lapply(ards_data, function(data) {
    glmer(formula = as.formula(formula_text),
            data = data,
            family = binomial(link = "logit"),
            control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e6)))
})
message('Model fit!')

<h2> Get fixed effects </h2>

In [ ]:
ards6_fe <- pool_fixed_effects(ards6_model)
ards6_fe_table <- create_fe_table(
  fe_df = ards6_fe,
  title = "**ARDS-6 Model - Pooled Fixed Effects**",
  output_file = 'ards6.html'
)

<h2> Figure 3: ards6 fixed effects </h2>

In [ ]:
ards6_fe_plot <- create_forest_plot(
  data = ards6_fe,
  main_breaks = c(0, 1, 2, 3, 4, 5),
  main_limits = c(0, 5),
  year_breaks = c(0, 1, 2, 3, 4, 5, 6),
  year_limits = c(0, 6),
  filename = "figure3_ards6_fixed_effects.tiff"
)

<h2> Get random effects <h2>

In [ ]:
ards6_re <- lapply(ards6_model, extract_random_effects)

<h2> Figure 2. Plot proportion set tidal volume and mean provider AOR </h2>

In [ ]:
primary_a = ltvv6_proportion_plot(ards6_data[[1]])
primary_b = random_effect_aor(
    data = ards6_re[[1]],
    re_group_name = 'prov_npi_shifted',
    aor_col_name = 'AOR',
    aor_ci_low_name = 'CI_lower',
    aor_ci_up_name = 'CI_upper',
    plot_title = 'Variation in Provider LTVV Adherence (ARDS-6)',
    x_axis_name = 'Provider',
    figure_name = 'ards6_aor.tiff'
)
primary_combined = (primary_a + primary_b) + plot_annotation(tag_levels = 'A')
ggsave('figure2_ards6_two_panel.tiff', plot = primary_combined, dpi = 600, width = 12, height = 6)

In [ ]:
summarize_model(ards6_model, ards6_data[[1]], 'ltvv_6', 'ards-6 Model')

<h2> ICU Type Adjustment: Before/After MOR and ICC </h2>

In [ ]:
# Before/after ICU-type adjustment: MOR and ICC comparison (Task 2 output)
explanatory_vars_no_icu <- setdiff(explanatory_vars, "icu_type")
formula_no_icu <- paste('ltvv_6 ~', paste(explanatory_vars_no_icu, collapse = ' + '),
                        '+ (1 | prov_npi_shifted)')
message(paste('No-ICU formula:', formula_no_icu))

ards6_no_icu_model <- future_lapply(ards_data, function(df) {
  glmer(as.formula(formula_no_icu), data = df, family = binomial('logit'),
        control = glmerControl(optimizer = 'bobyqa', optCtrl = list(maxfun = 2e6)))
})

summary_no_icu   <- summarize_model(ards6_no_icu_model, ards_data[[1]], 'ltvv_6', 'ards-6 Without ICU Type')
summary_with_icu <- summarize_model(ards6_model,         ards_data[[1]], 'ltvv_6', 'ards-6 With ICU Type')
create_model_summary_html(list(summary_no_icu, summary_with_icu),
                          output_file = 'ards6_icu_type_comparison.html')

<h1> e-Figure 1. ards6: Secondary analysis (Initial day of MV and subsequent days of MV)
 </h1>


<h2> Fit ards6 initial model </h2>


In [ ]:
explanatory_vars <- c("age", "race_category", "sex_category", "height_cm", "recorded_year", "elix_vw", "hospital_id", "icu_type", 'bmi_calc',
                      'sf_ratio', 'pco2', 'ph', 'laps2')

random_effect <- "prov_npi_shifted"
outcome <- 'ltvv_6'


# Set model seed
set.seed(42) 
formula_text <- paste(outcome,"~", paste(explanatory_vars, collapse = " + "), " + (1 | ", random_effect, ")")
message(paste('Formula:',formula_text))

message('Fitting model...')
ards6_initial_model <- future_lapply(initial_ards_data, function(data) {
    glmer(formula = as.formula(formula_text),
            data = data,
            family = binomial(link = "logit"),
            control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e6)))
})
message('Model fit!')

<h2> Fit ards6 subsequent model </h2>


In [ ]:
explanatory_vars <- c("age", "race_category", "sex_category", "height_cm", "recorded_year", "elix_vw", "hospital_id", "icu_type", 'bmi_calc',
                      'sf_ratio', 'pco2', 'ph', 'laps2')

random_effect <- "prov_npi_shifted"
outcome <- 'ltvv_6'


# Set model seed
set.seed(42) 
formula_text <- paste(outcome,"~", paste(explanatory_vars, collapse = " + "), " + (1 | ", random_effect, ")")
message(paste('Formula:',formula_text))

message('Fitting model...')
ards6_subsequent_model <- future_lapply(subsequent_ards_data, function(data) {
    glmer(formula = as.formula(formula_text),
            data = data,
            family = binomial(link = "logit"),
            control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e6)))
})
message('Model fit!')

<h2> Get fixed effects </h2>


In [ ]:
# Initial ards6
ards6_initial_fe <- pool_fixed_effects(ards6_initial_model)
ards6_initial_table <- create_fe_table(
  fe_df = ards6_initial_fe,
  title = "**ARDS-6 Initial Model - Pooled Fixed Effects**",
  output_file = 'ards6_initial_model.html'
)

# Subsequent ards6
ards6_subsequent_fe <- pool_fixed_effects(ards6_subsequent_model)
ards6_subsequent_table <- create_fe_table(
  fe_df = ards6_subsequent_fe,
  title = "**ARDS-6 Subsequent Model - Pooled Fixed Effects**",
  output_file = 'ards6_subsequent_model.html'
)

<h2> Plot fixed effects </h2>


In [ ]:
# Initial ards6
ards6_initial_fe_plot <- create_forest_plot(
  data = ards6_initial_fe,
  main_breaks = c(0, 5, 10, 15),
  main_limits = c(0, 15),
  year_breaks = c(0, 5, 10, 15),
  year_limits = c(0, 15),
  filename = "ards6_initial_fixed_effects.tiff"
)

# Subsequent ards6
ards6_subsequent_fe_plot <- create_forest_plot(
  data = ards6_subsequent_fe,
  main_breaks = c(0, 1, 2, 3, 4, 5),
  main_limits = c(0, 5),
  year_breaks = c(0, 1, 2, 3, 4, 5),
  year_limits = c(0, 5),
  filename = "ards6_subsequent_fixed_effects.tiff"
)

<h2> Get random effects </h2>

In [ ]:
ards6_initial_re <- lapply(ards6_initial_model, extract_random_effects)
ards6_subsequent_re <- lapply(ards6_subsequent_model, extract_random_effects)

<h2> e-Figure 1. Initial and subsequent ards6 </h2>

In [ ]:
# Initial
initial_a = ltvv6_proportion_plot(
  initial_ards_data[[1]],
  plot_title = 'Proportion of Set Tidal Volume by Provider\n(Initial, ARDS-6)'
)
initial_b = random_effect_aor(
    data = ards6_initial_re[[1]],
    re_group_name = 'prov_npi_shifted',
    aor_col_name = 'AOR',
    aor_ci_low_name = 'CI_lower',
    aor_ci_up_name = 'CI_upper',
    plot_title = 'Variation in Provider LTVV Adherence\n(Initial, ARDS-6)',
    x_axis_name = 'Provider',
    y_min = 0,
    y_max = 6,
    figure_name = 'ards6_initial_aor.tiff'
)

# Subsequent
subsequent_a = ltvv6_proportion_plot(
  subsequent_ards_data[[1]],
  plot_title = 'Proportion of Set Tidal Volume by Provider\n(Subsequent, ARDS-6)'
)
subsequent_b = random_effect_aor(
    data = ards6_subsequent_re[[1]],
    re_group_name = 'prov_npi_shifted',
    aor_col_name = 'AOR',
    aor_ci_low_name = 'CI_lower',
    aor_ci_up_name = 'CI_upper',
    plot_title = 'Variation in Provider LTVV Adherence\n(Subsequent, ARDS-6)',
    x_axis_name = 'Provider',
    y_min = 0,
    y_max = 6,
    figure_name = 'ards6_subsequent_aor.tiff'
)

# 4 panel
initial_sub_combined <- (
  (initial_a + initial_b) /    # Row 1
  (subsequent_a + subsequent_b)  # Row 2
) + plot_annotation(tag_levels = 'A')  # Auto-labels A, B, C, D
# Save combined figure
ggsave('e-Figure 1_ards6_initial_subsequent_4panel.tiff', plot = initial_sub_combined, dpi = 600, width = 12, height = 12)


<h1> e-Figure 2: ards8 - Tidal Volume Set by Provider </h1>
(A) Variation in LPV Adherence by Provider  (B) Provider Random Effects


<h2> Fit ards8 model </h2>


In [ ]:
# ards8 model setup
explanatory_vars <- c('age', 'race_category', 'sex_category', 'height_cm', 'recorded_year', 'elix_vw', 'hospital_id', 'icu_type', 'bmi_calc',
                      'sf_ratio', 'pco2', 'ph', 'laps2')
random_effect <- 'prov_npi_shifted'
outcome <- 'ltvv_8'

set.seed(42)
formula_text <- paste(outcome,'~', paste(explanatory_vars, collapse = ' + '), ' + (1 | ', random_effect, ')')
message(paste('Formula:', formula_text))

ards8_model <- future_lapply(ards_data, function(data) {
  glmer(formula = as.formula(formula_text),
        data    = data,
        family  = binomial(link = 'logit'),
        control = glmerControl(optimizer = 'bobyqa', optCtrl = list(maxfun = 2e6)))
})
message('ards8 model fit complete')


<h2> Get fixed effects </h2>


In [ ]:
ards8_fe <- pool_fixed_effects(ards8_model)
ards8_fe_table <- create_fe_table(
  fe_df = ards8_fe,
  title = '**ARDS-8 Model - Pooled Fixed Effects**',
  output_file = 'ards8.html'
)


<h2> Plot fixed effects </h2>


In [ ]:
ards8_fe_plot <- create_forest_plot(
  data = ards8_fe,
  main_breaks = c(0, 2, 4, 6),
  main_limits = c(0, 7),
  year_breaks = c(0, 5, 10, 15, 20),
  year_limits = c(0, 20),
  filename = 'ards8_fixed_effects.tiff'
)

<h2> Get random effects </h2>


In [ ]:
ards8_re <- lapply(ards8_model, extract_random_effects)

<h2> Plot proportion set tidal volume and mean provider AOR </h2>


In [ ]:
ards8_panel_a <- ltvv8_proportion_plot(ards8_data[[1]], plot_title = 'Provider Tidal Volume Distribution')
ards8_panel_b <- random_effect_aor(
  data            = ards8_re[[1]],
  re_group_name   = 'prov_npi_shifted',
  aor_col_name    = 'AOR',
  aor_ci_low_name = 'CI_lower',
  aor_ci_up_name  = 'CI_upper',
  plot_title      = 'Variation in Provider LTVV Adherence (ARDS-8)',
  x_axis_name     = 'Provider',
  figure_name     = 'e-Figure 2_ards8_provider_random_effects.tiff'
)
ards8_two_panel <- (ards8_panel_a + ards8_panel_b) + plot_annotation(tag_levels = 'A')
ggsave('e-Figure 2_ards8_two_panel.tiff', plot = ards8_two_panel, dpi = 600, width = 12, height = 6)


In [ ]:
summarize_model(ards8_model, ards8_data[[1]], 'ltvv_8', 'ards-8 Model')

<h1> e-Figure 3: ards8 - Initial and Subsequent Days </h1>


<h2> Fit ards8 initial model </h2>


In [ ]:
explanatory_vars <- c('age', 'race_category', 'sex_category', 'height_cm', 'recorded_year', 'elix_vw', 'hospital_id', 'icu_type', 'bmi_calc',
                      'sf_ratio', 'pco2', 'ph', 'laps2')
random_effect <- 'prov_npi_shifted'
outcome <- 'ltvv_8'

set.seed(42)
initial_ards8_formula <- paste(outcome,'~', paste(explanatory_vars, collapse = ' + '), ' + (1 | ', random_effect, ')')
message(paste('Initial ards8 formula:', initial_ards8_formula))

initial_ards8_model <- future_lapply(initial_ards_data, function(data) {
  glmer(formula = as.formula(initial_ards8_formula),
        data    = data,
        family  = binomial(link = 'logit'),
        control = glmerControl(optimizer = 'bobyqa', optCtrl = list(maxfun = 2e6)))
})
message('Initial ards8 model fit complete')


<h2> Fit ards8 subsequent model </h2>


In [ ]:
set.seed(42)
subsequent_ards8_formula <- paste(outcome,'~', paste(explanatory_vars, collapse = ' + '), ' + (1 | ', random_effect, ')')
message(paste('Subsequent ards8 formula:', subsequent_ards8_formula))

subsequent_ards8_model <- future_lapply(subsequent_ards_data, function(data) {
  glmer(formula = as.formula(subsequent_ards8_formula),
        data    = data,
        family  = binomial(link = 'logit'),
        control = glmerControl(optimizer = 'bobyqa', optCtrl = list(maxfun = 2e6)))
})
message('Subsequent ards8 model fit complete')

<h2> Get fixed effects </h2>


In [ ]:
initial_ards8_fe    <- pool_fixed_effects(initial_ards8_model)
subsequent_ards8_fe <- pool_fixed_effects(subsequent_ards8_model)

initial_ards8_table <- create_fe_table(
  fe_df = initial_ards8_fe,
  title = '**Initial ARDS-8 Model - Pooled Fixed Effects**',
  output_file = 'ards8_initial_model.html'
)

subsequent_ards8_table <- create_fe_table(
  fe_df = subsequent_ards8_fe,
  title = '**Subsequent ARDS-8 Model - Pooled Fixed Effects**',
  output_file = 'ards8_subsequent_model.html'
)


<h2> Plot fixed effects </h2>


In [ ]:
initial_ards8_fe_plot <- create_forest_plot(
  data = initial_ards8_fe,
  main_breaks = c(0, 2, 4, 6, 8),
  main_limits = c(0, 8),
  year_breaks = c(0, 10, 20, 30, 40),
  year_limits = c(0, 45),
  filename = 'ards8_initial_fixed_effects.tiff'
)

subsequent_ards8_fe_plot <- create_forest_plot(
  data = subsequent_ards8_fe,
  main_breaks = c(0, 2, 4, 6, 8),
  main_limits = c(0, 9),
  year_breaks = c(0, 5, 10, 15),
  year_limits = c(0, 15),
  filename = 'ards8_subsequent_fixed_effects.tiff'
)


<h2> Get random effects </h2>


In [ ]:
initial_ards8_re    <- lapply(initial_ards8_model, extract_random_effects)
subsequent_ards8_re <- lapply(subsequent_ards8_model, extract_random_effects)


<h2> Plot proportion set tidal volume and mean provider AOR </h2>


In [ ]:
initial_ards8_a <- ltvv8_proportion_plot(
  initial_ards_data[[1]],
  plot_title = 'Proportion of Set Tidal Volume by Provider\n(Initial, ARDS-8)')
initial_ards8_b <- random_effect_aor(
  data = initial_ards8_re[[1]],
  re_group_name = 'prov_npi_shifted',
  aor_col_name = 'AOR',
  aor_ci_low_name = 'CI_lower',
  aor_ci_up_name = 'CI_upper',
  plot_title = 'Variation in Provider LTVV Adherence\n(Initial, ARDS-8)',
  x_axis_name = 'Provider',
  y_min = 0,
  y_max = 2,
  figure_name = 'ards8_initial_aor.tiff'
)

subsequent_ards8_a <- ltvv8_proportion_plot(
  subsequent_ards_data[[1]],
  plot_title = 'Proportion of Set Tidal Volume by Provider\n(Subsequent, ARDS-8)')
subsequent_ards8_b <- random_effect_aor(
  data = subsequent_ards8_re[[1]],
  re_group_name = 'prov_npi_shifted',
  aor_col_name = 'AOR',
  aor_ci_low_name = 'CI_lower',
  aor_ci_up_name = 'CI_upper',
  plot_title = 'Variation in Provider LTVV Adherence\n(Subsequent, ARDS-8)',
  x_axis_name = 'Provider',
  y_min = 0,
  y_max = 16,
  figure_name = 'ards8_subsequent_aor.tiff'
)

ards8_initial_sub_combined <- ((initial_ards8_a + initial_ards8_b) / (subsequent_ards8_a + subsequent_ards8_b)) + plot_annotation(tag_levels = 'A')
ggsave('e-Figure 3_ards8_initial_subsequent_4panel.tiff', plot = ards8_initial_sub_combined, dpi = 600, width = 12, height = 12)

<h1> e-Figure 6: ards6 - SARS-CoV-2 Sensitivity </h1>
(A) Variation in LPV Adherence by Provider  (B) Provider Random Effects


<h2> Fit ards6 COVID sensitivity model </h2>


In [ ]:
explanatory_vars <- c('age', 'race_category', 'sex_category', 'height_cm', 'recorded_year', 'elix_vw', 'hospital_id', 'icu_type', 'bmi_calc',
                      'sf_ratio', 'pco2', 'ph', 'laps2', 'covid')
random_effect <- 'prov_npi_shifted'
outcome <- 'ltvv_6'

set.seed(42)
covid_formula <- paste(outcome,'~', paste(explanatory_vars, collapse = ' + '), ' + (1 | ', random_effect, ')')
message(paste('COVID sensitivity formula:', covid_formula))

covid_model <- future_lapply(ards6_data, function(data) {
  glmer(formula = as.formula(covid_formula),
        data    = data,
        family  = binomial(link = 'logit'),
        control = glmerControl(optimizer = 'bobyqa', optCtrl = list(maxfun = 2e6)))
})
message('COVID sensitivity model fit complete')


<h2> Get fixed effects </h2>


In [ ]:
covid_fe <- pool_fixed_effects(covid_model)
covid_fe_table <- create_fe_table(
  fe_df = covid_fe,
  title = '**ARDS-6 COVID Sensitivity Model - Pooled Fixed Effects**',
  output_file = 'ards6_covid_model.html'
)

<h2> Plot fixed effects </h2>


In [ ]:
covid_fe_plot <- create_forest_plot(
  data = covid_fe,
  main_breaks = c(0, 1, 2, 3, 4),
  main_limits = c(0, 4),
  year_breaks = c(0, 1, 2, 3, 4, 5),
  year_limits = c(0, 5),
  filename = 'ards6_covid_fixed_effects.tiff'
)


<h2> Get random effects </h2>


In [ ]:
covid_re <- lapply(covid_model, extract_random_effects)

<h2> Plot proportion set tidal volume and mean provider AOR </h2>


In [ ]:
covid_panel_a <- ltvv6_proportion_plot(
  ards6_data[[1]],
  plot_title = 'Proportion of Set Tidal Volume by Provider'
)

covid_panel_b <- random_effect_aor(
  data            = covid_re[[1]],
  re_group_name   = 'prov_npi_shifted',
  aor_col_name    = 'AOR',
  aor_ci_low_name = 'CI_lower',
  aor_ci_up_name  = 'CI_upper',
  plot_title      = 'Variation in Provider LTVV Adherence\n(ARDS-6, SARS-CoV-2)',
  x_axis_name     = 'Provider',
  figure_name     = 'e-Figure 6_ards6_covid_provider_random_effects.tiff'
)
covid_two_panel <- (covid_panel_a + covid_panel_b) + plot_annotation(tag_levels = 'A')
ggsave('e-Figure 6_ards6_covid_two_panel.tiff', plot = covid_two_panel, dpi = 600, width = 12, height = 6)

<h1> e-Figure 4: MV8 - Secondary analysis </h1>
(A) Variation in LPV Adherence by Provider  (B) Provider Random Effects


<h2> Fit MV8 model </h2>


In [ ]:
explanatory_vars <- c('age', 'race_category', 'sex_category', 'height_cm', 'recorded_year', 'elix_vw', 'hospital_id', 'icu_type', 'bmi_calc',
                      'sf_ratio', 'pco2', 'ph', 'laps2')
random_effect <- 'prov_npi_shifted'
outcome <- 'ltvv_8'

set.seed(42)
mv8_formula <- paste(outcome,'~', paste(explanatory_vars, collapse = ' + '), ' + (1 | ', random_effect, ')')
message(paste('MV8 formula:', mv8_formula))

mv8_model <- future_lapply(mv_data, function(data) {
  glmer(formula = as.formula(mv8_formula),
        data    = data,
        family  = binomial(link = 'logit'),
        control = glmerControl(optimizer = 'bobyqa', optCtrl = list(maxfun = 2e6)))
})
message('MV8 model fit complete')

<h2> Get fixed effects </h2>


In [ ]:
mv8_fe <- pool_fixed_effects(mv8_model)
mv8_fe_table <- create_fe_table(
  fe_df = mv8_fe,
  title = '**MV-8 Model - Pooled Fixed Effects**',
  output_file = 'MV8.html'
)


<h2> Plot fixed effects </h2>


In [ ]:
mv8_fe_plot <- create_forest_plot(
  data = mv8_fe,
  main_breaks = c(0, 2, 4),
  main_limits = c(0, 4),
  year_breaks = c(0, 5, 10, 15, 20),
  year_limits = c(0, 20),
  filename = 'mv8_fixed_effects.tiff'
)


<h2> Get random effects </h2>


In [ ]:
mv8_re <- lapply(mv8_model, extract_random_effects)


<h2> Plot proportion set tidal volume and mean provider AOR </h2>


In [ ]:
mv8_panel_a <- ltvv8_proportion_plot(
  mv_data[[1]],
  plot_title = 'Proportion of Set Tidal Volume by Provider')
mv8_panel_b <- random_effect_aor(
  data            = mv8_re[[1]],
  re_group_name   = 'prov_npi_shifted',
  aor_col_name    = 'AOR',
  aor_ci_low_name = 'CI_lower',
  aor_ci_up_name  = 'CI_upper',
  plot_title      = 'Variation in Provider LTVV Adherence (MV-8)',
  x_axis_name     = 'Provider',
  figure_name     = 'e-Figure 4_mv8_provider_random_effects.tiff'
)
mv8_two_panel <- (mv8_panel_a + mv8_panel_b) + plot_annotation(tag_levels = 'A')
ggsave('e-Figure 4_mv8_two_panel.tiff', plot = mv8_two_panel, dpi = 600, width = 12, height = 6)


<h1> e-Figure 5: MV8 - Initial and Subsequent Days </h1>


<h2> Fit MV8 initial model </h2>


In [ ]:
explanatory_vars <- c('age', 'race_category', 'sex_category', 'height_cm', 'recorded_year', 'elix_vw', 'hospital_id', 'icu_type', 'bmi_calc',
                      'sf_ratio', 'pco2', 'ph', 'laps2')
random_effect <- 'prov_npi_shifted'
outcome <- 'ltvv_8'

set.seed(42)
initial_mv8_formula <- paste(outcome,'~', paste(explanatory_vars, collapse = ' + '), ' + (1 | ', random_effect, ')')
message(paste('Initial MV8 formula:', initial_mv8_formula))

initial_mv8_model <- future_lapply(initial_mv_data, function(data) {
  glmer(formula = as.formula(initial_mv8_formula),
        data    = data,
        family  = binomial(link = 'logit'),
        control = glmerControl(optimizer = 'bobyqa', optCtrl = list(maxfun = 2e6)))
})
message('Initial MV8 model fit complete')


<h2> Fit MV8 subsequent model </h2>


In [ ]:
set.seed(42)
subsequent_mv8_formula <- paste(outcome,'~', paste(explanatory_vars, collapse = ' + '), ' + (1 | ', random_effect, ')')
message(paste('Subsequent MV8 formula:', subsequent_mv8_formula))

subsequent_mv8_model <- future_lapply(subsequent_mv_data, function(data) {
  glmer(formula = as.formula(subsequent_mv8_formula),
        data    = data,
        family  = binomial(link = 'logit'), 
        control = glmerControl(optimizer = 'bobyqa', optCtrl = list(maxfun = 2e6)))
})
message('Subsequent MV8 model fit complete')


<h2> Get fixed effects </h2>


In [ ]:
initial_mv8_fe    <- pool_fixed_effects(initial_mv8_model)
subsequent_mv8_fe <- pool_fixed_effects(subsequent_mv8_model)

initial_mv8_table <- create_fe_table(
  fe_df = initial_mv8_fe,
  title = '**Initial MV-8 Model - Pooled Fixed Effects**',
  output_file = 'MV8_initial_model.html'
)

subsequent_mv8_table <- create_fe_table(
  fe_df = subsequent_mv8_fe,
  title = '**Subsequent MV-8 Model - Pooled Fixed Effects**',
  output_file = 'MV8_subsequent_model.html'
)


<h2> Plot fixed effects </h2>


In [ ]:
initial_mv8_fe_plot <- create_forest_plot(
  data = initial_mv8_fe,
  main_breaks = c(0, 2, 4, 6),
  main_limits = c(0, 6),
  year_breaks = c(0, 10, 20, 30, 40),
  year_limits = c(0, 40),
  filename = 'mv8_initial_fixed_effects.tiff'
)

subsequent_mv8_fe_plot <- create_forest_plot(
  data = subsequent_mv8_fe,
  main_breaks = c(0, 2, 4),
  main_limits = c(0, 5),
  year_breaks = c(0, 5, 10, 15, 20),
  year_limits = c(0, 20),
  filename = 'mv8_subsequent_fixed_effects.tiff'
)


<h2> Get random effects </h2>


In [ ]:
initial_mv8_re    <- lapply(initial_mv8_model, extract_random_effects)
subsequent_mv8_re <- lapply(subsequent_mv8_model, extract_random_effects)

<h2> Plot proportion set tidal volume and mean provider AOR </h2>


In [ ]:
initial_mv8_a <- ltvv8_proportion_plot(
  initial_mv_data[[1]],
  plot_title = 'Proportion of Set Tidal Volume by Provider\n(Initial, MV-8)')
initial_mv8_b <- random_effect_aor(
  data = initial_mv8_re[[1]],
  re_group_name = 'prov_npi_shifted',
  aor_col_name = 'AOR',
  aor_ci_low_name = 'CI_lower',
  aor_ci_up_name = 'CI_upper',
  plot_title = 'Variation in Provider LTVV Adherence\n(Initial, MV-8)',
  x_axis_name = 'Provider',
  y_min = 0,
  y_max = 4,
  figure_name = 'mv8_initial_aor.tiff'
)

subsequent_mv8_a <- ltvv8_proportion_plot(
  subsequent_mv_data[[1]],
  plot_title = 'Proportion of Set Tidal Volume by Provider\n(Subsequent, MV-8)')
subsequent_mv8_b <- random_effect_aor(
  data = subsequent_mv8_re[[1]],
  re_group_name = 'prov_npi_shifted',
  aor_col_name = 'AOR',
  aor_ci_low_name = 'CI_lower',
  aor_ci_up_name  = 'CI_upper',
  plot_title = 'Variation in Provider LTVV Adherence\n(Subsequent, MV-8)',
  x_axis_name = 'Provider',
  y_min = 0,
  y_max = 4,
  figure_name = 'mv8_subsequent_aor.tiff'
)

mv8_initial_sub_combined <- ((initial_mv8_a + initial_mv8_b) / (subsequent_mv8_a + subsequent_mv8_b)) + plot_annotation(tag_levels = 'A')
ggsave('e-Figure 5_mv8_initial_subsequent_4panel.tiff', plot = mv8_initial_sub_combined, dpi = 600, width = 12, height = 12)


In [ ]:
# Summarize all models and write combined HTML report
ards6_summary <- summarize_model(ards6_model, ards6_data[[1]], 'ltvv_6', 'ards-6 Overall')
ards6_initial_summary <- summarize_model(ards6_initial_model, initial_ards_data[[1]], 'ltvv_6', 'ards-6 Initial')
ards6_subsequent_summary <- summarize_model(ards6_subsequent_model, subsequent_ards_data[[1]], 'ltvv_6', 'ards-6 Subsequent')
covid_summary <- summarize_model(covid_model, ards6_data[[1]], 'ltvv_6', 'ards-6 + COVID')

ards8_summary <- summarize_model(ards8_model, ards8_data[[1]], 'ltvv_8', 'ards-8 Overall')
initial_ards8_summary <- summarize_model(initial_ards8_model, initial_ards_data[[1]], 'ltvv_8', 'ards-8 Initial')
subsequent_ards8_summary <- summarize_model(subsequent_ards8_model, subsequent_ards_data[[1]], 'ltvv_8', 'ards-8 Subsequent')

mv8_summary <- summarize_model(mv8_model, mv_data[[1]], 'ltvv_8', 'MV-8 Overall')
initial_mv8_summary <- summarize_model(initial_mv8_model, initial_mv_data[[1]], 'ltvv_8', 'MV-8 Initial')
subsequent_mv8_summary <- summarize_model(subsequent_mv8_model, subsequent_mv_data[[1]], 'ltvv_8', 'MV-8 Subsequent')

model_summaries <- list(
  ards6_summary,
  ards6_initial_summary,
  ards6_subsequent_summary,
  covid_summary,
  ards8_summary,
  initial_ards8_summary,
  subsequent_ards8_summary,
  mv8_summary,
  initial_mv8_summary,
  subsequent_mv8_summary
)

create_model_summary_html(model_summaries, output_file = 'table2_model_summary.html')
